# Model Building on the Other stem first

In [4]:
import os
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
import tensorflow as tf
#libraries for building the model
from tensorflow.keras.layers import Conv2D,MaxPool2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [8]:
##loading the other data(mel's) and lables
other_data = np.load(f'processed_data/other_features.npy')
other_labels = np.load(f'processed_data/other_labels.npy')

In [9]:
print(other_data.shape)
print(other_labels.shape)

(10198, 150, 150, 1)
(10198,)


## 3 way data split
80% Training: To teach the model.

10% Validation: To check for "Overfitting" during the training process.

10% Test: To give a final, unbiased accuracy score at the very end.

In [10]:
# shuffling is need cause we added the data in a sequential order first bollypop then carnatic so data inside data is in order
#stratify keep the proportion of data same in both test and train
from sklearn.model_selection import train_test_split

# this single line splits and shuffles at the same time
X_train, X_temp, Y_train, Y_temp = train_test_split(
    other_data,
    other_labels,
    test_size=0.2,
    random_state=42,
    shuffle=True, # be default it is true
    stratify=other_labels # Ensures 20% of EACH genre goes to the test set
)


#20 % remaining data in temp
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True, # be default it is true
    stratify= Y_temp # Ensures 20% of EACH genre goes to the test set
)
# Summary of the "Shuffling" Timeline
# During Split: Handled by train_test_split(shuffle=True). (Do this now)
# During Training: Handled by model.fit(shuffle=True). (Do this later)
# now ready for normalization

In [11]:
print(X_train.shape)
print(Y_train.shape)
print(X_val.shape)
print(Y_val.shape)
print(X_test.shape)
print(Y_test.shape)

(8158, 150, 150, 1)
(8158,)
(1020, 150, 150, 1)
(1020,)
(1020, 150, 150, 1)
(1020,)


## data normalization

In [12]:
other_data[:1]

# the values are in decibels(power_to_db) when converted to mel scale
# need to normalize the value form 0 to 1

array([[[[-48.452854],
         [-48.11557 ],
         [-49.23192 ],
         ...,
         [-47.95298 ],
         [-44.208584],
         [-39.58476 ]],

        [[-48.672886],
         [-43.46166 ],
         [-42.037247],
         ...,
         [-47.11456 ],
         [-45.67823 ],
         [-43.559166]],

        [[-54.899876],
         [-43.74206 ],
         [-40.584118],
         ...,
         [-45.917923],
         [-43.058113],
         [-40.126766]],

        ...,

        [[-79.950966],
         [-80.      ],
         [-80.      ],
         ...,
         [-80.      ],
         [-74.35068 ],
         [-63.332966]],

        [[-80.      ],
         [-80.      ],
         [-80.      ],
         ...,
         [-80.      ],
         [-74.50041 ],
         [-63.504673]],

        [[-80.      ],
         [-80.      ],
         [-80.      ],
         ...,
         [-80.      ],
         [-74.47734 ],
         [-63.471592]]]], dtype=float32)

## Normalization from (-80 - 0)  to  0 - 1 
The "Golden Rule": Split First, Normalize Second
Many beginners make the mistake of normalizing the entire dataset (all 10,000 samples) and then splitting it. This causes Data Leakage.
Why? Normalization (like Min-Max Scaling) uses the min and max values of your data. If you normalize the whole set before splitting, your training data "knows" the maximum value of the test set. This "future information" leaks into your model, making your training accuracy look artificially high while your real-world performance fails.
The Correct Way: 
1.  Split your data into $X_{train}$ and $X_{test}$.
2.  Calculate the min and max using only $X_{train}$.
3.  Apply that same scaling to both $X_{train}$ and $X_{test}$.

In [13]:
# 1. Calculate stats from TRAINING data only
# This represents the "known world" to the model
# min and max of x_train only
X_min = X_train.min()
X_max = X_train.max()

# 2. Apply the same scaling to all three sets
# This squishes all dB values into a 0.0 to 1.0 range
X_train = (X_train - X_min) / (X_max - X_min)
X_val = (X_val - X_min) / (X_max - X_min)
X_test = (X_test - X_min) / (X_max - X_min)

# 3. Verification - Essential Check!
print("--- Normalization Audit ---")
print(f"Train Range: {X_train.min()} to {X_train.max()}") # Must be 0.0 to 1.0
print(f"Val Range:   {X_val.min():.2f} to {X_val.max():.2f}")
print(f"Test Range:  {X_test.min():.2f} to {X_test.max():.2f}")

--- Normalization Audit ---
Train Range: 0.0 to 1.0
Val Range:   0.00 to 1.00
Test Range:  0.00 to 1.00


### One hot encoding when to use for labels 

In [14]:
# In your current project, you actually don't need to do one-hot encoding because of the specific loss function we chose. However, understanding when to use it is a "rite of passage" for any AI developer.

# Here is the breakdown of when to use it and when to skip it.

# 1. The "Loss Function" Rule
# The decision to use one-hot encoding is tied directly to which Loss Function you pick in model.compile():

# Scenario A: Integer Labels (0, 1, 2, 3, 4)

# Use: loss='sparse_categorical_crossentropy'

# Decision: Do NOT one-hot encode.

# Why: Keras handles the math internally. This is more memory-efficient because you aren't creating a huge matrix of zeros. (This is your current setup).

# Scenario B: One-Hot Labels ([1,0,0,0,0], [0,1,0,0,0]...)

# Use: loss='categorical_crossentropy'

# Decision: Yes, one-hot encode.

# Why: You use this when you want the model to output "probabilities" that exactly match a target vector.

## Building the CNN Architecture

link : https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks/

In [15]:
# consists of convulations, pooling, flatening(fully connected), dense and dropout
# using the sequential modle(layer stacking)
# allows group layers together and the output of one layers is the input for other layer

model = tf.keras.models.Sequential()


In [16]:
classes = ['bollypop', 'carnatic', 'ghazal', 'semiclassical', 'sufi']

In [17]:
mel_shape = X_train[0].shape
#it is just the one melspectrogram(one chunk out of multiple chunks of one audio file)
#and it has 210*210 - 44100 elements

## Adding the layers

In [18]:
model.add(Conv2D(filters = 32, kernel_size= 3, padding= 'same', activation= 'relu', input_shape = mel_shape ))

model.add(Conv2D(filters= 32, kernel_size = 3, activation = 'relu'))

model.add(MaxPool2D(pool_size=2, strides = 2))


          

In [19]:
model.add(Conv2D(filters= 64, kernel_size = 3, padding = 'same', activation= 'relu')) #input shape will be there 
                                                                                    # same as first layers(no need to give again)
model.add(Conv2D(filters = 64, kernel_size = 3, activation= 'relu'))

model.add(MaxPool2D(pool_size = 2, strides = 2))

In [20]:
model.add(Conv2D(filters= 128, kernel_size = 3, padding = 'same', activation= 'relu')) #input shape will be there 
                                                                                    # same as first layers(no need to give again)
model.add(Conv2D(filters = 128, kernel_size = 3, activation= 'relu'))

model.add(MaxPool2D(pool_size = 2, strides = 2))

In [21]:
# drop some features to overcome overfitting

model.add(Dropout(0.3)) #-> dropping 30% neurons

In [22]:
model.add(Conv2D(filters= 256, kernel_size = 3, padding = 'same', activation= 'relu')) #input shape will be there 
                                                                                    # same as first layers(no need to give again)
model.add(Conv2D(filters = 256, kernel_size = 3, activation= 'relu'))

model.add(MaxPool2D(pool_size = 2, strides = 2))

In [23]:
model.add(Conv2D(filters= 512, kernel_size = 3, padding = 'same', activation= 'relu')) #input shape will be there 
                                                                                    # same as first layers(no need to give again)
model.add(Conv2D(filters = 512, kernel_size = 3, activation= 'relu'))

model.add(MaxPool2D(pool_size = 2, strides = 2))

In [24]:
model.add(Dropout(0.3))

In [25]:
model.add(Flatten())
# Fully Connected (FC)The fully connected layer (FC) operates on a flattened input where each input is 
# connected to all neurons. If present, FC layers are usually found towards the end of CNN architectures
# and can be used to optimize objectives such as class scores.

In [26]:
#constructing the hidden layers, units are neuron and value activation func is relu(as sigmoid is costly(may be on CPU resourses)
model.add(Dense(units = 1200, activation = 'relu'))

In [27]:
model.add(Dropout(0.45))

In [28]:
#final output layer
model.add(Dense(units=len(classes), activation = 'softmax'))

In [29]:
model.summary()
# The parameters here are the extracted features from the mel's by conv2d(extraction), pooling(max,avg value extraction)
# and the dropout(removal) again repeating and increasing filters , hence it just increases

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 150, 150, 32)      320       
                                                                 
 conv2d_1 (Conv2D)           (None, 148, 148, 32)      9248      
                                                                 
 max_pooling2d (MaxPooling2D  (None, 74, 74, 32)       0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 74, 74, 64)        18496     
                                                                 
 conv2d_3 (Conv2D)           (None, 72, 72, 64)        36928     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 36, 36, 64)       0         
 2D)                                                    

## Compiling the Model

In [30]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])
#loss = sparse_categorical_crossentropy, no one hot encoding needed

## Training the Model

In [31]:
training_history = model.fit(X_train, Y_train, epochs= 30, batch_size= 32, validation_data=(X_val, Y_val))

Epoch 1/30
255/255 [==============================] - 52s 154ms/step - loss: 1.5858 - accuracy: 0.2380 - val_loss: 1.5272 - val_accuracy: 0.3118
Epoch 2/30
255/255 [==============================] - 30s 117ms/step - loss: 1.4454 - accuracy: 0.3655 - val_loss: 1.3245 - val_accuracy: 0.4186
Epoch 3/30
255/255 [==============================] - 30s 119ms/step - loss: 1.3052 - accuracy: 0.4516 - val_loss: 1.2021 - val_accuracy: 0.5078
Epoch 4/30
255/255 [==============================] - 30s 117ms/step - loss: 1.1863 - accuracy: 0.5205 - val_loss: 1.1673 - val_accuracy: 0.5382
Epoch 5/30
255/255 [==============================] - 28s 109ms/step - loss: 1.0720 - accuracy: 0.5721 - val_loss: 1.0266 - val_accuracy: 0.5980
Epoch 6/30
255/255 [==============================] - 25s 98ms/step - loss: 0.9368 - accuracy: 0.6288 - val_loss: 0.8133 - val_accuracy: 0.6833
Epoch 7/30
255/255 [==============================] - 25s 99ms/step - loss: 0.8237 - accuracy: 0.6760 - val_loss: 0.7882 - val_accu

In [33]:
# Save in the modern Keras format
model.save('indian_music_classifier_v1.keras') 
print("Model saved as indian_music_classifier_v1.keras")

Model saved as indian_music_classifier_v1.keras
